In [1]:
import requests
import os
import json
import time

In [ ]:
def get_with_retry(url, params, max_retries = 3):
    wait = 2

    for attempt in range(1, max_retries +1):
        try:
            response = requests.get(url=url, params = params, timeout = 10)
            response.raise_for_status()
            return response.json()
        
        except requests.exceptions.HTTPError as e:
            status = e.response.status_code

            if status == 429:
                retry_after = int(e.response.headers.get("Retry-After", wait))
                print(f"Rate limited. Waiting {retry_after}s (attempt {attempt}/{max_retries})")
            
            elif status >= 500:
                print(f"Server error {status}. Waiting {wait}s (attempt {attempt}/{max_retries})")
                time.sleep(wait)
                wait *= 2

            else:
                raise

        except requests.exceptions.ConnectionError:
            print(f"Connection error. Waiting {wait}s (attempt {attempt}/{max_retries})")
            time.smeep(wait)
            wait *= 2

    raise Exception(f"Failed after {max_retries} attempts")

In [ ]:
import requests
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

@retry(
    stop = stop_after_attempt(3)
    wait = wait_exponential(multiplier = 1, min=2, max= 16),
    retry = retry_if_exception_type(requests.exceptions.HTTPError)
)
def get_with_retry(url, params):
    response = requests.get(url=url, params=params, timeout=10)
    response.raise_for_status()
    return response.json()


In [2]:
import requests
import time

def get_with_retry(url, params, max_retries=3):
    wait = 2

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url=url, params=params, timeout=10)
            response.raise_for_status()
            return response.json()

        except requests.exceptions.HTTPError as e:
            status = e.response.status_code

            if status == 429:
                retry_after = int(e.response.headers.get("Retry-After", wait))
                print(f"Rate limited. Waiting {retry_after}s (attempt {attempt}/{max_retries})")
                time.sleep(retry_after)

            elif status >= 500:
                print(f"Server error {status}. Waiting {wait}s (attempt {attempt}/{max_retries})")
                time.sleep(wait)
                wait *= 2

            else:
                raise

        except requests.exceptions.ConnectionError:
            print(f"Connection error. Waiting {wait}s (attempt {attempt}/{max_retries})")
            time.sleep(wait)
            wait *= 2

    raise Exception(f"Failed after {max_retries} attempts")


all_characters = []
page = 1

while True:
    data = get_with_retry(
        url="https://rickandmortyapi.com/api/character",
        params={"page": page}
    )

    all_characters.extend(data["results"])
    print(f"Fetched page {page} — {len(data['results'])} characters")

    if not data["info"]["next"]:
        break

    page += 1

print(f"Total characters fetched: {len(all_characters)}")

Fetched page 1 — 20 characters
Fetched page 2 — 20 characters
Fetched page 3 — 20 characters
Fetched page 4 — 20 characters
Fetched page 5 — 20 characters
Fetched page 6 — 20 characters
Fetched page 7 — 20 characters
Fetched page 8 — 20 characters
Fetched page 9 — 20 characters
Fetched page 10 — 20 characters
Fetched page 11 — 20 characters
Fetched page 12 — 20 characters
Fetched page 13 — 20 characters
Fetched page 14 — 20 characters
Fetched page 15 — 20 characters
Fetched page 16 — 20 characters
Fetched page 17 — 20 characters
Fetched page 18 — 20 characters
Fetched page 19 — 20 characters
Fetched page 20 — 20 characters
Fetched page 21 — 20 characters
Fetched page 22 — 20 characters
Fetched page 23 — 20 characters
Fetched page 24 — 20 characters
Fetched page 25 — 20 characters
Fetched page 26 — 20 characters
Fetched page 27 — 20 characters
Fetched page 28 — 20 characters
Fetched page 29 — 20 characters
Fetched page 30 — 20 characters
Rate limited. Waiting 10s (attempt 1/3)
Fetched p

In [5]:
import requests
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

@retry(
    stop = stop_after_attempt(3),
    wait = wait_exponential(multiplier = 1, min=2, max= 16),
    retry = retry_if_exception_type(requests.exceptions.HTTPError)
)
def get_with_retry(url,params):
    response = requests.get(url=url,params=params,timeout=10)
    response.raise_for_status()
    return response.json()

all_characters = []
page = 1

while True:
    data = get_with_retry(
        url = "https://rickandmortyapi.com/api/character",
        params = {"page":page}
    )

    all_characters.extend(data["results"])
    print(f"Fetched page {page} - {len(data['results'])} characters")

    if not data["info"]["next"]:
        break

    page += 1

print(f"Total characters fetched: {len(all_characters)}")

Fetched page 1 - 20 characters
Fetched page 2 - 20 characters
Fetched page 3 - 20 characters
Fetched page 4 - 20 characters
Fetched page 5 - 20 characters
Fetched page 6 - 20 characters
Fetched page 7 - 20 characters
Fetched page 8 - 20 characters
Fetched page 9 - 20 characters
Fetched page 10 - 20 characters
Fetched page 11 - 20 characters
Fetched page 12 - 20 characters
Fetched page 13 - 20 characters
Fetched page 14 - 20 characters
Fetched page 15 - 20 characters
Fetched page 16 - 20 characters
Fetched page 17 - 20 characters
Fetched page 18 - 20 characters
Fetched page 19 - 20 characters
Fetched page 20 - 20 characters
Fetched page 21 - 20 characters
Fetched page 22 - 20 characters
Fetched page 23 - 20 characters
Fetched page 24 - 20 characters
Fetched page 25 - 20 characters
Fetched page 26 - 20 characters
Fetched page 27 - 20 characters
Fetched page 28 - 20 characters
Fetched page 29 - 20 characters
Fetched page 30 - 20 characters


RetryError: RetryError[<Future at 0x1fcc2aca210 state=finished raised HTTPError>]

In [7]:
import requests
import time
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type, before_sleep

def wait_for_rate_limit(retry_state):
    exc = retry_state.outcome.exception()
    if hasattr(exc, 'response') and exc.response.status_code == 429:
        wait = int(exc.response.headers.get("Retry-After", 60))
        print(f"Rate limited on page. Waiting {wait}s...")
        time.sleep(wait)
    else:
        print(f"Retrying after error... attempt {retry_state.attempt_number}")

@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=2, max=16),
    retry=retry_if_exception_type(requests.exceptions.HTTPError),
    before_sleep=wait_for_rate_limit
)
def get_with_retry(url, params):
    response = requests.get(url=url, params=params, timeout=10)
    response.raise_for_status()
    return response.json()

all_characters = []
page = 1

while True:
    data = get_with_retry(
        url="https://rickandmortyapi.com/api/character",
        params={"page": page}
    )

    all_characters.extend(data["results"])
    print(f"Fetched page {page} - {len(data['results'])} characters")

    if not data["info"]["next"]:
        break

    page += 1

print(f"Total characters fetched: {len(all_characters)}")

Fetched page 1 - 20 characters
Fetched page 2 - 20 characters
Fetched page 3 - 20 characters
Fetched page 4 - 20 characters
Fetched page 5 - 20 characters
Fetched page 6 - 20 characters
Fetched page 7 - 20 characters
Fetched page 8 - 20 characters
Fetched page 9 - 20 characters
Fetched page 10 - 20 characters
Fetched page 11 - 20 characters
Fetched page 12 - 20 characters
Fetched page 13 - 20 characters
Fetched page 14 - 20 characters
Fetched page 15 - 20 characters
Fetched page 16 - 20 characters
Fetched page 17 - 20 characters
Fetched page 18 - 20 characters
Fetched page 19 - 20 characters
Fetched page 20 - 20 characters
Fetched page 21 - 20 characters
Fetched page 22 - 20 characters
Fetched page 23 - 20 characters
Fetched page 24 - 20 characters
Fetched page 25 - 20 characters
Fetched page 26 - 20 characters
Fetched page 27 - 20 characters
Fetched page 28 - 20 characters
Fetched page 29 - 20 characters
Fetched page 30 - 20 characters
Rate limited on page. Waiting 10s...
Fetched page